In [1]:
data_file_path="../finance_artifact/data_ingestion/feature_store/finance_complaint"

In [2]:
from finance_complaint.config.spark_manager import spark_session

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 06:43:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import os

print(os.path.exists(data_file_path))
print(os.path.abspath(data_file_path))

True
/app/finance_artifact/data_ingestion/feature_store/finance_complaint


In [4]:
df = spark_session.read.parquet(data_file_path)

In [5]:
df.count()

1720278

In [6]:
from pyspark.sql.functions import col

In [6]:
df.select('complaint_id')

DataFrame[complaint_id: string]

## Conf

In [7]:
def update_column_attribute(df):
    for column in df.columns:
        setattr(df,column,column)

In [8]:
update_column_attribute(df)

## Printing unique values in each column

In [9]:
for column in df.columns:
    print(f"{column}:{df.select(column).distinct().count()}")

company:5520
company_public_response:11
company_response:7


complaint_id:1717536


complaint_what_happened:646623
consumer_consent_provided:6
consumer_disputed:3
date_received:1826
date_sent_to_company:1893
issue:160
product:18
state:64
sub_issue:223
sub_product:76
submitted_via:6
tags:4
timely:2
zip_code:25724


In [10]:
complaint_table="complaint"
df.createOrReplaceTempView(complaint_table)

In [9]:
sql = spark_session.sql

In [10]:
n_row = df.count()

In [11]:
#Target column
df.groupBy(df.consumer_disputed).count().collect()

AttributeError: 'DataFrame' object has no attribute 'consumer_disputed'

In [1]:
df.printSchema()

NameError: name 'df' is not defined

In [15]:
missing_target_df = sql(f"select * from {complaint_table} where {df.consumer_disputed} ='N/A' ")

In [16]:
df = sql(f"select * from {complaint_table} where {df.consumer_disputed} <>'N/A' ")

In [17]:
complaint_table="complaint_temp"
df.createOrReplaceTempView(complaint_table)

In [18]:
update_column_attribute(df)

In [19]:
duplicate_df = df.groupBy(df.columns).count().filter("count > 1")

duplicate_df.show()

+--------------------+-----------------------+--------------------+------------+-----------------------+-------------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----+--------------------+--------------------+-------------+--------------+------+--------+-----+
|             company|company_public_response|    company_response|complaint_id|complaint_what_happened|consumer_consent_provided|consumer_disputed|       date_received|date_sent_to_company|               issue|             product|state|           sub_issue|         sub_product|submitted_via|          tags|timely|zip_code|count|
+--------------------+-----------------------+--------------------+------------+-----------------------+-------------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----+--------------------+--------------------+-------------+--------------+------+--------+-----+
|Arm

In [20]:
df= df.dropDuplicates()

In [21]:

def perform_null_analysis(df,table_name):
    null_value_analysis=[]
    n_row=df.count()
    for column in df.columns:
        print(column)
        response = sql(f"select {n_row} as  total_row,count(*) as null_row_{column},(count(*)*100)/{n_row} as missing_percentage,'{column}' as  column_name from {table_name} where {column} is null").collect()
        null_value_analysis.append(response)
    return null_value_analysis

In [22]:
null_report = perform_null_analysis(df,complaint_table)

company
company_public_response


company_response
complaint_id
complaint_what_happened
consumer_consent_provided
consumer_disputed
date_received
date_sent_to_company
issue
product
state
sub_issue
sub_product
submitted_via
tags
timely
zip_code


In [23]:
def unwanted_column_by_missing_percentage(null_value_analysis,per_thres=20):
    columns= []
    for row in null_value_analysis:
        row_info=row[0]
        if row_info.missing_percentage>per_thres:
            print(row_info)
            columns.append(row_info.column_name)

    return columns


In [24]:
columns = unwanted_column_by_missing_percentage(null_value_analysis=null_report)

Row(total_row=236431, null_row_company_public_response=217056, missing_percentage=91.80522012764824, column_name='company_public_response')
Row(total_row=236431, null_row_sub_issue=213539, missing_percentage=90.31768253739992, column_name='sub_issue')
Row(total_row=236431, null_row_sub_product=153327, missing_percentage=64.85063295422343, column_name='sub_product')
Row(total_row=236431, null_row_tags=379289, missing_percentage=160.42270260668016, column_name='tags')


In [25]:
def drop_column(df,columns):
    selected_column = list(filter(lambda x:x not in columns,df.columns))
    selected_column = ",".join(selected_column)
    df= sql(f"select {selected_column}  from {complaint_table} ")
    return df

In [26]:
df=drop_column(df,columns)

In [27]:
columns = perform_null_analysis(df,complaint_table)

company
company_response
complaint_id
complaint_what_happened
consumer_consent_provided
consumer_disputed
date_received
date_sent_to_company
issue
product
state
submitted_via
timely
zip_code


In [28]:
unwanted_column_by_missing_percentage(columns)

[]

In [29]:
#dropping feature as we have found more than 20% of null value in above columns

In [30]:
df.collect()

[Row(company='Navient Solutions, LLC.', company_response='Closed with explanation', complaint_id='2383556', complaint_what_happened='', consumer_consent_provided='Consent not provided', consumer_disputed='No', date_received='2017-03-13T12:00:00-05:00', date_sent_to_company='2017-03-15T12:00:00-05:00', issue="Can't repay my loan", product='Student loan', state=None, submitted_via='Web', timely='Yes', zip_code='XXXXX'),
 Row(company='BlueChip Financial', company_response='Closed with explanation', complaint_id='2399949', complaint_what_happened='XXXX is the collection company handling to collection. They called Tuesday both mine and my husbands phone. My husband returned their call and they discussed the debt and what is was going to take to fix it. My husband is not on the loan or my personal account. I spoke with them Wednesday about the fact that they discussed my personal info with my husband she informed me they are able to. She stated that I woild be served papers today if not paid

In [31]:
## Unique values in each columns



In [32]:
print(f"Total number of row: {df.count()}")
for column in df.columns:
    print(f"{column}:{df.select(column).distinct().count()}")

Total number of row: 442146


company:3263
company_response:5


complaint_id:236223


complaint_what_happened:96353
consumer_consent_provided:6
consumer_disputed:2
date_received:419
date_sent_to_company:545
issue:95
product:13
state:63
submitted_via:6
timely:2
zip_code:18056


In [33]:
update_column_attribute(df)

In [34]:
df=drop_column(df,columns=[df.complaint_id])

In [35]:
update_column_attribute(df)

In [36]:
df.printSchema()

root
 |-- company: string (nullable = true)
 |-- company_response: string (nullable = true)
 |-- complaint_what_happened: string (nullable = true)
 |-- consumer_consent_provided: string (nullable = true)
 |-- consumer_disputed: string (nullable = true)
 |-- date_received: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- product: string (nullable = true)
 |-- state: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- timely: string (nullable = true)
 |-- zip_code: string (nullable = true)



In [37]:
print(f"Total number of row: {df.count()}")
for column in df.columns:
    print(f"{column}:{df.select(column).distinct().count()}")

Total number of row: 442146


company:3263
company_response:5


complaint_what_happened:96353
consumer_consent_provided:6
consumer_disputed:2
date_received:419


date_sent_to_company:545


issue:95


product:13
state:63
submitted_via:6
timely:2
zip_code:18056


In [38]:
df.groupBy(df.company_response).count().show()
df.groupBy(df.consumer_consent_provided).count().show()
df.groupBy(df.consumer_disputed).count().show()
df.groupBy(df.product).count().show()
df.groupBy(df.submitted_via).count().show()
df.groupBy(df.timely).count().show()


+--------------------+------+
|    company_response| count|
+--------------------+------+
|   Untimely response|  1793|
|Closed with non-m...| 51592|
|Closed with monet...| 26754|
|Closed with expla...|353683|
|              Closed|  8324|
+--------------------+------+

+-------------------------+------+
|consumer_consent_provided| count|
+-------------------------+------+
|                     null|    33|
|        Consent withdrawn|     7|
|                    Other|  8466|
|         Consent provided|184684|
|     Consent not provided|142382|
|                      N/A|106574|
+-------------------------+------+

+-----------------+------+
|consumer_disputed| count|
+-----------------+------+
|               No|368588|
|              Yes| 73558|
+-----------------+------+

+--------------------+------+
|             product| count|
+--------------------+------+
|     Debt collection| 93143|
|    Virtual currency|    18|
|         Payday loan|  3420|
|     Money transfers|  3227|
|Chec

+-------------+------+
|submitted_via| count|
+-------------+------+
|        Phone| 25482|
|          Fax|  5767|
|     Referral| 50121|
|  Postal mail| 25194|
|          Web|335572|
|        Email|    10|
+-------------+------+

+------+------+
|timely| count|
+------+------+
|    No| 15121|
|   Yes|427025|
+------+------+



In [89]:

df.groupBy(df.company_response).count().collect()
df.groupBy(df.consumer_consent_provided).count().collect()
df.groupBy(df.consumer_disputed).count().collect()
df.groupBy(df.product).count().collect()
df.groupBy(df.submitted_via).count().collect()
# df.groupBy(df.timely).count().collect()


[Row(submitted_via='Phone', count=25482),
 Row(submitted_via='Fax', count=5767),
 Row(submitted_via='Referral', count=50121),
 Row(submitted_via='Postal mail', count=25194),
 Row(submitted_via='Web', count=335572),
 Row(submitted_via='Email', count=10)]

In [39]:
df.groupBy(df.product).count().collect()

[Row(product='Debt collection', count=93143),
 Row(product='Virtual currency', count=18),
 Row(product='Payday loan', count=3420),
 Row(product='Money transfers', count=3227),
 Row(product='Checking or savings account', count=6),
 Row(product='Mortgage', count=84921),
 Row(product='Prepaid card', count=2805),
 Row(product='Credit reporting', count=102649),
 Row(product='Consumer Loan', count=21977),
 Row(product='Credit card', count=47258),
 Row(product='Bank account or service', count=48873),
 Row(product='Other financial service', count=1040),
 Row(product='Student loan', count=32809)]

In [41]:
ONE_HOT_FEATURE = [df.company_response,df.consumer_consent_provided,df.submitted_via,df.timely]
BINARY_ENCODING = [df.product,]
TARGET_FEATURE = [df.consumer_disputed]

#df.company_response No null value
#df.consumer_consent_provided  replace null with top category 
#df.consumer_disputed target feature label encoding
#df.product one hot encoding 

#df.product no null value


In [42]:
remaining_column = list(filter(lambda x: x not in ONE_HOT_FEATURE+BINARY_ENCODING+TARGET_FEATURE,df.columns))

In [43]:
remaining_column

['company',
 'complaint_what_happened',
 'date_received',
 'date_sent_to_company',
 'issue',
 'state',
 'zip_code']

In [44]:
df.groupBy(df.company).count().count()

3263

In [45]:
FREQUENCY_ENCODING = [df.company]

In [46]:
df.select(df.company).count()-df.select(df.company).dropna().count()


0

In [47]:
REPLACE_NULL_WITH_TOP_VALUE = [df.zip_code,df.state,df.consumer_consent_provided]

In [48]:
REPLACE_NULL_WITH_TOP_VALUE

['zip_code', 'state', 'consumer_consent_provided']

In [49]:
print(f"Total number of row: {df.count()}")
for column in remaining_column:
    print(f"{column}:  {df.select(column).distinct().count()}")

Total number of row: 442146
company:  3263


complaint_what_happened:  96353
date_received:  419
date_sent_to_company:  545
issue:  95
state:  63
zip_code:  18056


In [50]:
df.groupBy(df.issue).count().show()

+--------------------+-----+
|               issue|count|
+--------------------+-----+
|Communication tac...|11656|
|Application proce...|  306|
|Advertising and m...| 1577|
|Balance transfer fee|  107|
|Customer service/...|  256|
|        Adding money|  102|
|Credit card prote...| 1403|
|Closing/Cancellin...| 3367|
|Received a loan I...|  402|
|                Fees|  187|
|Can't stop charge...|  331|
|          Bankruptcy|  147|
|Forbearance / Wor...|  207|
|Credit determination| 1526|
|Loan modification...|32002|
|    Cash advance fee|   81|
|Other transaction...| 1023|
|Customer service ...| 2092|
|      Getting a loan|  364|
|  Delinquent account| 1809|
+--------------------+-----+
only showing top 20 rows



In [51]:
remaining_column

['company',
 'complaint_what_happened',
 'date_received',
 'date_sent_to_company',
 'issue',
 'state',
 'zip_code']

In [54]:
df.columns

['company',
 'company_response',
 'complaint_what_happened',
 'consumer_consent_provided',
 'consumer_disputed',
 'date_received',
 'date_sent_to_company',
 'issue',
 'product',
 'state',
 'submitted_via',
 'timely',
 'zip_code']

In [59]:
df.select(df.complaint_what_happened).collect()[3]

Row(complaint_what_happened='Convergant Outsourcing LLC has reported a paid collection in which I did pay to settle the account with them for {$250.00} because I owed XXXX {$280.00} for a closed unpaid phone bill. Which was in XX/XX/XXXX when it was sent to collection for none payment..Convergent Outsourcing LLC has the account listed as opened on XX/XX/XXXX which is not accurate')

In [57]:
sql(f"select {df.complaint_what_happened} from {complaint_table} limit 5").collect()

[Row(complaint_what_happened=''),
 Row(complaint_what_happened='XXXX is the collection company handling to collection. They called Tuesday both mine and my husbands phone. My husband returned their call and they discussed the debt and what is was going to take to fix it. My husband is not on the loan or my personal account. I spoke with them Wednesday about the fact that they discussed my personal info with my husband she informed me they are able to. She stated that I woild be served papers today if not paid and then garnishment would start soon after.'),
 Row(complaint_what_happened=''),
 Row(complaint_what_happened='Convergant Outsourcing LLC has reported a paid collection in which I did pay to settle the account with them for {$250.00} because I owed XXXX {$280.00} for a closed unpaid phone bill. Which was in XX/XX/XXXX when it was sent to collection for none payment..Convergent Outsourcing LLC has the account listed as opened on XX/XX/XXXX which is not accurate'),
 Row(complaint_w

In [60]:
TOKENIZER_FEATURE = [df.complaint_what_happened]

In [61]:
ONE_HOT_FEATURE

['company_response', 'consumer_consent_provided', 'submitted_via', 'timely']

In [62]:
FREQUENCY_ENCODING = [df.company,df.issue,df.state,df.zip_code]

In [63]:
TARGET_FEATURE

['consumer_disputed']

In [64]:
TOKENIZER_FEATURE

['complaint_what_happened']

In [65]:
remaining_column

['company',
 'complaint_what_happened',
 'date_received',
 'date_sent_to_company',
 'issue',
 'state',
 'zip_code']

In [66]:
from pyspark.sql.types import TimestampType

In [67]:
df=df.withColumn(df.date_received,col(df.date_received).cast(TimestampType()))

In [68]:
update_column_attribute(df)

In [69]:
df=df.withColumn(df.date_sent_to_company,col(df.date_sent_to_company).cast(TimestampType()))
df.printSchema()

root
 |-- company: string (nullable = true)
 |-- company_response: string (nullable = true)
 |-- complaint_what_happened: string (nullable = true)
 |-- consumer_consent_provided: string (nullable = true)
 |-- consumer_disputed: string (nullable = true)
 |-- date_received: timestamp (nullable = true)
 |-- date_sent_to_company: timestamp (nullable = true)
 |-- issue: string (nullable = true)
 |-- product: string (nullable = true)
 |-- state: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- timely: string (nullable = true)
 |-- zip_code: string (nullable = true)



In [70]:
df.select([df.date_received,df.date_sent_to_company]).show()

+-------------------+--------------------+
|      date_received|date_sent_to_company|
+-------------------+--------------------+
|2017-03-13 17:00:00| 2017-03-15 17:00:00|
|2017-03-23 17:00:00| 2017-03-23 17:00:00|
|2017-03-22 17:00:00| 2017-03-23 17:00:00|
|2017-03-31 17:00:00| 2017-03-31 17:00:00|
|2017-03-29 17:00:00| 2017-03-29 17:00:00|
|2017-03-15 17:00:00| 2017-03-16 17:00:00|
|2017-04-18 17:00:00| 2017-04-19 17:00:00|
|2017-03-09 17:00:00| 2017-03-09 17:00:00|
|2017-04-12 17:00:00| 2017-04-13 17:00:00|
|2017-04-11 17:00:00| 2017-04-11 17:00:00|
|2017-03-06 17:00:00| 2017-03-06 17:00:00|
|2017-03-20 17:00:00| 2017-03-20 17:00:00|
|2017-04-03 17:00:00| 2017-04-04 17:00:00|
|2017-04-05 17:00:00| 2017-04-09 17:00:00|
|2017-03-06 17:00:00| 2017-03-06 17:00:00|
|2017-03-28 17:00:00| 2017-03-28 17:00:00|
|2017-03-22 17:00:00| 2017-03-27 17:00:00|
|2017-03-10 17:00:00| 2017-03-10 17:00:00|
|2017-03-07 17:00:00| 2017-03-09 17:00:00|
|2017-04-20 17:00:00| 2017-04-21 17:00:00|
+----------

In [71]:
from pyspark.sql.types import  LongType

In [73]:
df = df.withColumn("diff_in_days",(col("date_sent_to_company").cast(LongType())-col("date_received").cast(LongType()))/(60*60*24))

In [74]:
update_column_attribute(df)

In [75]:
remove_column = [df.date_received,df.date_sent_to_company]

In [76]:
df=df.drop(col(df.date_received)).drop(col(df.date_sent_to_company))

In [77]:
NUMERICAL_FEATURE = [df.diff_in_days,]
ONE_HOT_FEATURE+\
FREQUENCY_ENCODING+\
BINARY_ENCODING+\
TARGET_FEATURE

['company_response',
 'consumer_consent_provided',
 'submitted_via',
 'timely',
 'company',
 'issue',
 'state',
 'zip_code',
 'product',
 'consumer_disputed']

In [78]:
BINARY_ENCODING

['product']

In [79]:
FREQUENCY_ENCODING=FREQUENCY_ENCODING+BINARY_ENCODING

In [90]:
FREQUENCY_ENCODING.remove('issue')

In [91]:
FREQUENCY_ENCODING

['company', 'state', 'zip_code', 'product']

In [ ]:
ONE_HOT_FEATURE

['company_response', 'consumer_consent_provided', 'submitted_via', 'timely']

26/05/26 09:13:07 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 9313465 ms exceeds timeout 120000 ms
26/05/26 09:13:08 WARN SparkContext: Killing executors is not supported by current scheduler.
